# Excise 1

##### E01: train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; Did it improve over a bigram model?

##### E01： 訓練一個 trigram（三字詞）語言模型，也就是根據前兩個字元來預測下一個字元。請分別用「手動計數矩陣」和「單層神經網路」兩種方法實作，並評估它的 Loss


In [78]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

In [37]:
# 首先讀取檔案
words=open('./names.txt','r').read().splitlines()

# 建立字碼對應表
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

V = len(itos)

In [55]:
T = torch.zeros((V**2, V),dtype=torch.int32)
for w in words:
  chs = ['.','.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    ix3 = stoi[ch3]

    T[ix1*V + ix2, ix3] += 1



In [ ]:
P = (T+1).float()
P /= P.sum(1, keepdim=True)

log_likelihood = 0.0
n = 0

for w in words:
#for w in ["andrejq"]:
  chs = ['.', '.'] + list(w) + ['.']
  for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    ix3 = stoi[ch3]
    prob = P[ix1 * V + ix2, ix3]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n}')

log_likelihood=tensor(-504653.)
nll=tensor(504653.)
2.2119739055633545


In [ ]:
# 單層神經網路

xs, ys = [], []
for w in words:  
    chs = ['.', '.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        ix3 = stoi[ch3]

        xs.append(ix1 * V + ix2)
        ys.append(ix3)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()

# 初始化網路
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((V**2, V), generator=g, requires_grad=True)

for k in range(2000):
    # forward pass
    #xenc = F.one_hot(xs, num_classes=V**2).float()
    #logits = xenc @ W
    # 這裡有一個簡單的寫法
    logits = W[xs] # 這裡的 logits 是一個 (num, V) 的矩陣，代表每個樣本對應到每個字元的 logit 值
    counts = logits.exp() 
    probs = counts / counts.sum(1, keepdim=True) # 修正為 keepdim
    
    # 這裡的 loss 同時包含了 NLL 和 L2 正則化
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W**2).mean()
    
    # backward pass
    W.grad = None
    loss.backward()

    # update
    W.data += -50 * W.grad

    if k % 100 == 0: # 每 10 次印一次就好，畫面比較乾淨
        print(f"Step {k}: Loss = {loss.item():.4f}")

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

##### E02: split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see?

##### E02： 將原本神經網路的輸入劃分為「訓練集（Training Set）」、「驗證集（Validation Set）」和「測試集（Test Set）」（比例為 80%/10%/10%），並在訓練集上優化，最後在驗證集和測試集上評估 Loss。

In [63]:
'.'.join('.')

'.'